In [4]:
# ============================================================
# NOTEBOOK 2 : ÉCONOMÉTRIE TEMPORELLE
# VAR mensuel, Causalité de Granger, IRF, FEVD
# ============================================================
# Question : Le pétrole drive-t-il l'intégralité de la relation
# commerciale Russie-Chine, ou existe-t-il un facteur géopolitique
# autonome (effet "sanctions/pivot") ?
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from pathlib import Path
from datetime import datetime
import warnings

warnings.filterwarnings("ignore")

# Statsmodels
from statsmodels.tsa.api import VAR
from statsmodels.tsa.stattools import adfuller, kpss, grangercausalitytests
from statsmodels.tsa.vector_ar.vecm import coint_johansen
from statsmodels.stats.diagnostic import acorr_ljungbox
import statsmodels.api as sm

# Config
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update(
    {
        "figure.figsize": (14, 6),
        "font.size": 11,
        "axes.titlesize": 14,
        "axes.labelsize": 12,
        "figure.dpi": 120,
    }
)

# Chemins robustes : detecte la racine du projet (Chine-russie)
_candidates = [Path.cwd(), Path.cwd().parent]
for c in _candidates:
    _proj = c / "russia_china_dependency"
    if _proj.exists() and (_proj / "01_raw_data").exists():
        PROJECT_ROOT = c
        break
else:
    PROJECT_ROOT = Path.cwd()

PROJECT = PROJECT_ROOT / "russia_china_dependency"
RAW = PROJECT / "01_raw_data"
PROCESSED = PROJECT / "02_processed_data"

print(f"Setup OK | Racine: {PROJECT_ROOT}")

Setup OK | Racine: d:\Users\Proprietaire\Desktop\Projet_perso\Chine-russie


In [5]:
# ============================================================
# CHARGEMENT & CONSTRUCTION DU PANEL MENSUEL
# Sources : Comtrade (commerce) + FRED (Brent, USD/RUB, USD/CNY)
# ============================================================

# --- 1. Commerce mensuel Comtrade ---
ct = pd.read_csv(
    RAW / "trade" / "comtrade_detailed_2013_2024.csv",
    usecols=[
        "freqCode",
        "period",
        "flowCode",
        "reporterISO",
        "partnerISO",
        "primaryValue",
        "aggrLevel",
    ],
)
ct_monthly = ct[ct["freqCode"] == "M"].copy()

# Agréger tous les produits par mois pour obtenir le total bilatéral
trade_monthly = (
    ct_monthly.groupby(["period", "flowCode", "reporterISO", "partnerISO"])[
        "primaryValue"
    ]
    .sum()
    .reset_index()
)

# Pivoter : exports CHN→RUS et imports CHN←RUS
chn_exp = trade_monthly[
    (trade_monthly["reporterISO"] == "CHN")
    & (trade_monthly["partnerISO"] == "RUS")
    & (trade_monthly["flowCode"] == "X")
][["period", "primaryValue"]].rename(columns={"primaryValue": "CHN_exports_to_RUS"})

chn_imp = trade_monthly[
    (trade_monthly["reporterISO"] == "CHN")
    & (trade_monthly["partnerISO"] == "RUS")
    & (trade_monthly["flowCode"] == "M")
][["period", "primaryValue"]].rename(columns={"primaryValue": "CHN_imports_from_RUS"})

# Convertir period (YYYYMM) en datetime
for df_tmp in [chn_exp, chn_imp]:
    df_tmp["date"] = pd.to_datetime(df_tmp["period"].astype(str), format="%Y%m")

trade = (
    chn_exp[["date", "CHN_exports_to_RUS"]]
    .merge(chn_imp[["date", "CHN_imports_from_RUS"]], on="date", how="outer")
    .sort_values("date")
    .set_index("date")
)

# Convertir en milliards USD
trade = trade / 1e9
trade.columns = [
    "CHN_exp_to_RUS_bn",
    "RUS_exp_to_CHN_bn",
]  # Miroir : import CHN from RUS = export RUS to CHN

print(
    f"Commerce mensuel : {len(trade)} mois, {trade.index.min():%Y-%m} -> {trade.index.max():%Y-%m}"
)

# --- 2. Brent mensuel (FRED) ---
brent = pd.read_csv(RAW / "prices" / "brent_oil_daily.csv")
brent["date"] = pd.to_datetime(brent["observation_date"])
brent = brent[["date", "DCOILBRENTEU"]].set_index("date")
brent = brent.replace(".", np.nan).astype(float)
brent_m = brent.resample("MS").mean().rename(columns={"DCOILBRENTEU": "brent_usd"})

# --- 3. USD/RUB mensuel (prices ou macro selon disponibilite) ---
_rub_paths = [
    RAW / "prices" / "usd_rub_investing.csv",
    RAW / "macro" / "usd_rub_investing.csv",
]
rub_path = next((p for p in _rub_paths if p.exists()), None)
if rub_path is None:
    raise FileNotFoundError(
        f"USD/RUB non trouve. Placer usd_rub_investing.csv dans "
        f"01_raw_data/prices/ ou 01_raw_data/macro/"
    )
rub = pd.read_csv(rub_path)
rub["date"] = pd.to_datetime(rub["Date"], format="mixed", dayfirst=False)
rub = rub[["date", "Price"]].set_index("date").sort_index()
rub_m = rub.resample("MS").mean().rename(columns={"Price": "usd_rub"})

# --- 4. USD/CNY mensuel ---
cny = pd.read_csv(RAW / "prices" / "usd_cny_daily.csv")
cny["date"] = pd.to_datetime(cny["observation_date"])
cny = cny[["date", "DEXCHUS"]].set_index("date")
cny = cny.replace(".", np.nan).astype(float)
cny_m = cny.resample("MS").mean().rename(columns={"DEXCHUS": "usd_cny"})

# --- 5. Fusion ---
panel = trade.join([brent_m, rub_m, cny_m], how="inner").dropna()
panel = panel.asfreq("MS")  # Forcer fréquence mensuelle

print(f"\nPanel mensuel final : {len(panel)} observations")
print(f"Période : {panel.index.min():%Y-%m} -> {panel.index.max():%Y-%m}")
print(f"\nStatistiques descriptives :")
print(panel.describe().round(2))
panel.head()

Commerce mensuel : 96 mois, 2017-01 -> 2024-12


FileNotFoundError: [Errno 2] No such file or directory: 'd:\\Users\\Proprietaire\\Desktop\\Projet_perso\\Chine-russie\\russia_china_dependency\\01_raw_data\\prices\\usd_cny_daily.csv'

In [ ]:
# ============================================================
# VISUALISATION EXPLORATOIRE DES SÉRIES TEMPORELLES
# ============================================================

fig, axes = plt.subplots(3, 2, figsize=(16, 14), sharex=True)

# 1. Exports russes vers Chine
ax = axes[0, 0]
ax.plot(panel.index, panel["RUS_exp_to_CHN_bn"], color="#d62728", linewidth=1.5)
ax.axvline(
    pd.Timestamp("2022-02-24"),
    color="black",
    linestyle="--",
    alpha=0.7,
    label="Invasion Ukraine",
)
ax.axvline(
    pd.Timestamp("2020-03-01"), color="gray", linestyle=":", alpha=0.7, label="COVID-19"
)
ax.set_ylabel("Mrd USD")
ax.set_title("Exports Russie → Chine (imports CHN from RUS)")
ax.legend(fontsize=9)

# 2. Exports chinois vers Russie
ax = axes[0, 1]
ax.plot(panel.index, panel["CHN_exp_to_RUS_bn"], color="#1f77b4", linewidth=1.5)
ax.axvline(pd.Timestamp("2022-02-24"), color="black", linestyle="--", alpha=0.7)
ax.axvline(pd.Timestamp("2020-03-01"), color="gray", linestyle=":", alpha=0.7)
ax.set_ylabel("Mrd USD")
ax.set_title("Exports Chine → Russie")

# 3. Brent
ax = axes[1, 0]
ax.plot(panel.index, panel["brent_usd"], color="#2ca02c", linewidth=1.5)
ax.axvline(pd.Timestamp("2022-02-24"), color="black", linestyle="--", alpha=0.7)
ax.set_ylabel("USD/baril")
ax.set_title("Prix du Brent")

# 4. USD/RUB
ax = axes[1, 1]
ax.plot(panel.index, panel["usd_rub"], color="#9467bd", linewidth=1.5)
ax.axvline(pd.Timestamp("2022-02-24"), color="black", linestyle="--", alpha=0.7)
ax.set_ylabel("RUB par USD")
ax.set_title("Taux de change USD/RUB")
ax.invert_yaxis()  # Plus haut = rouble plus faible

# 5. Balance commerciale bilatérale
ax = axes[2, 0]
balance = panel["RUS_exp_to_CHN_bn"] - panel["CHN_exp_to_RUS_bn"]
colors = ["#d62728" if v > 0 else "#1f77b4" for v in balance]
ax.bar(panel.index, balance, width=25, color=colors, alpha=0.7)
ax.axhline(0, color="black", linewidth=0.5)
ax.axvline(pd.Timestamp("2022-02-24"), color="black", linestyle="--", alpha=0.7)
ax.set_ylabel("Mrd USD")
ax.set_title("Balance commerciale RUS-CHN (rouge = excédent russe)")

# 6. Matrice de corrélation
ax = axes[2, 1]
corr = panel.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr,
    mask=mask,
    annot=True,
    fmt=".2f",
    cmap="RdBu_r",
    center=0,
    vmin=-1,
    vmax=1,
    ax=ax,
    square=True,
    cbar_kws={"shrink": 0.8},
)
ax.set_title("Corrélations contemporaines")

plt.tight_layout()
plt.savefig("nb2_exploration_series_temporelles.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure sauvegardee : nb2_exploration_series_temporelles.png")

In [ ]:
# ============================================================
# TESTS DE STATIONNARITÉ
# Stratégie de confirmation : ADF (H0: racine unitaire) + KPSS (H0: stationnaire)
# Si les deux divergent → ambiguïté → tester en différence
# ============================================================


def stationarity_tests(series, name, alpha=0.05):
    """Test ADF + KPSS avec interprétation conjointe."""
    results = {}

    # ADF
    adf_stat, adf_p, adf_lags, adf_nobs, adf_crit, _ = adfuller(
        series.dropna(), autolag="AIC"
    )
    results["ADF_stat"] = adf_stat
    results["ADF_pval"] = adf_p
    results["ADF_lags"] = adf_lags
    adf_reject = adf_p < alpha  # Rejeter H0 = pas de racine unitaire = stationnaire

    # KPSS
    kpss_stat, kpss_p, kpss_lags, kpss_crit = kpss(
        series.dropna(), regression="c", nlags="auto"
    )
    results["KPSS_stat"] = kpss_stat
    results["KPSS_pval"] = kpss_p
    kpss_reject = kpss_p < alpha  # Rejeter H0 = pas stationnaire

    # Interprétation conjointe
    if adf_reject and not kpss_reject:
        verdict = "STATIONNAIRE"
    elif not adf_reject and kpss_reject:
        verdict = "NON-STATIONNAIRE (I(1))"
    elif adf_reject and kpss_reject:
        verdict = "AMBIGU (trend-stationnaire ?)"
    else:
        verdict = "AMBIGU (faible puissance)"

    results["verdict"] = verdict
    return results


# Tester en niveaux et en différences premières
print("=" * 80)
print(f"{'Variable':<25} {'ADF p-val':>10} {'KPSS p-val':>11} {'Verdict (niveau)':>25}")
print("=" * 80)

test_vars = {
    "RUS_exp_to_CHN_bn": "Exports RUS→CHN",
    "CHN_exp_to_RUS_bn": "Exports CHN→RUS",
    "brent_usd": "Brent (USD)",
    "usd_rub": "USD/RUB",
}

stationarity_results = {}
for col, label in test_vars.items():
    res = stationarity_tests(panel[col], col)
    stationarity_results[col] = res
    print(
        f"  {label:<23} {res['ADF_pval']:>10.4f} {res['KPSS_pval']:>11.4f} {res['verdict']:>25}"
    )

print("\n" + "=" * 80)
print(
    f"{'Variable (diff.)':<25} {'ADF p-val':>10} {'KPSS p-val':>11} {'Verdict (diff.)':>25}"
)
print("=" * 80)

for col, label in test_vars.items():
    diff = panel[col].diff().dropna()
    res = stationarity_tests(diff, f"d_{col}")
    stationarity_results[f"d_{col}"] = res
    print(
        f"  d({label}){'':<{18-len(label)}} {res['ADF_pval']:>10.4f} {res['KPSS_pval']:>11.4f} {res['verdict']:>25}"
    )

print("\n→ Les séries en NIVEAUX sont probablement I(1). On travaillera en DIFFÉRENCES")
print("  ou on testera la cointégration pour un VECM si pertinent.")

In [3]:
# ============================================================
# TEST DE COINTÉGRATION DE JOHANSEN
# Question : Existe-t-il une relation d'équilibre de long terme
# entre les exports RUS→CHN, le Brent et le taux de change ?
# ============================================================

# Sélection des variables pour le test
coint_vars = panel[["RUS_exp_to_CHN_bn", "brent_usd", "usd_rub"]].dropna()

# Johansen test (det_order : 0=pas de constante, 1=constante dans EC)
johansen_result = coint_johansen(coint_vars, det_order=0, k_ar_diff=2)

print("=" * 70)
print("TEST DE COINTÉGRATION DE JOHANSEN")
print(f"Variables : {list(coint_vars.columns)}")
print(f"Observations : {len(coint_vars)}, Lags : 2")
print("=" * 70)

print(
    f"\n{'Rang r':<10} {'Trace Stat':>12} {'CV 5%':>10} {'CV 1%':>10} {'Cointégré ?':>15}"
)
print("-" * 60)
for i in range(len(johansen_result.trace_stat)):
    trace = johansen_result.trace_stat[i]
    cv5 = johansen_result.trace_stat_crit_vals[i, 1]
    cv1 = johansen_result.trace_stat_crit_vals[i, 2]
    coint = "OUI ***" if trace > cv1 else ("OUI **" if trace > cv5 else "NON")
    print(f"  r={i:<6} {trace:>12.3f} {cv5:>10.3f} {cv1:>10.3f} {coint:>15}")

print(
    f"\n{'Rang r':<10} {'Max-Eigen':>12} {'CV 5%':>10} {'CV 1%':>10} {'Cointégré ?':>15}"
)
print("-" * 60)
for i in range(len(johansen_result.max_eig_stat)):
    maxeig = johansen_result.max_eig_stat[i]
    cv5 = johansen_result.max_eig_stat_crit_vals[i, 1]
    cv1 = johansen_result.max_eig_stat_crit_vals[i, 2]
    coint = "OUI ***" if maxeig > cv1 else ("OUI **" if maxeig > cv5 else "NON")
    print(f"  r={i:<6} {maxeig:>12.3f} {cv5:>10.3f} {cv1:>10.3f} {coint:>15}")

print("\n⚠ CAVEAT : N=96 est en-dessous des recommandations habituelles (N>100).")
print("  Les résultats sont exploratoires. Le VAR en différences est plus prudent.")

NameError: name 'panel' is not defined